# LLM04 Data and Model Poisoning — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM04 — Data and Model Poisoning | **Risk Severity**: High

This notebook:
1. **Uploads** all artifact files (scenarios, model-based checks) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM04 data and model poisoning test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks upsert).
Registered IDs are available in-memory for the evaluation steps below.

**Three detection scenarios**:
- **RAG Corpus Poisoning**: Tests whether adversarial content in the retrieval store manipulates outputs
- **Behavioral Drift**: Compares current responses against known-good baselines to detect silent changes
- **Backdoor/Sleeper Triggers**: Tests whether specific trigger phrases activate hidden backdoor behaviors

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_artifact,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM04-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  \u2713 Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

### Register Model-Based Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    print(f"Registering model-based check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  \u2713 Registered: {check_data['name']} (ID: {result.id})")

print(f"\nModel-based checks registered: {len(registered_checks)}")


### Artifact Upload Summary

In [ ]:
print("=" * 60)
print("LLM04 Data and Model Poisoning \u2014 Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  \u2022 {name} \u2192 {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  \u2022 {name} \u2192 {cid}")
print("\n\u2713 All artifacts ready. Proceeding to evaluation...")

---
## Part 2 — Run Evaluation

### Behavioral Drift: Baseline Capture Workflow

The **behavioral drift** scenario compares the agent's current responses against known-good baselines
stored in the `result` field of `behavioral-drift.jsonl`. Before running drift detection for the first
time, you need to capture a baseline:

1. **Point at your trusted agent**: Ensure `owasp/target.json` references the trusted model version.
2. **Run the standardized prompts**: Execute the prompts in `behavioral-drift.jsonl` against the trusted agent.
3. **Save as baseline**: Update each row's `result` field with the trusted response.
4. **Version the baseline**: Update `version` in `behavioral-drift_meta.md` (e.g., `"1.0.0"`).
5. **Commit**: The committed JSONL is your baseline source of truth.

To update the baseline after validating a new model version, repeat steps 1-5 and increment the version.

### Configuration

The target agent is loaded from the shared `owasp/target.json` file.
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

Each scenario is evaluated with **one specialized model-based check**:
- `LLM04-corpus-poisoning` → `LLM04-corpus-poisoning-detector`
- `LLM04-behavioral-drift` → `LLM04-behavioral-drift-detector`
- `LLM04-backdoor-trigger` → `LLM04-backdoor-trigger-detector`

In [ ]:
# Target loaded from owasp/target.json. To use a different config: target = build_target(CATEGORY_DIR, config_path="target.prod.json")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

SCENARIO_CHECK_MAP = {
    "LLM04-corpus-poisoning":  ["LLM04-corpus-poisoning-detector"],
    "LLM04-behavioral-drift":  ["LLM04-behavioral-drift-detector"],
    "LLM04-backdoor-trigger":  ["LLM04-backdoor-trigger-detector"],
}

### Single-Turn Tests — All Scenarios

Each scenario runs via `okareo.run_simulation()` constrained to `max_turns=1`.
A pass-through driver delivers the scenario input verbatim to the target agent; the agent's
response is evaluated by the scenario-specific model-based check.

In [ ]:
single_turn_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-single-turn-driver",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

all_results = {}

for scenario_name, scenario in registered_scenarios.items():
    checks_for_scenario = SCENARIO_CHECK_MAP.get(scenario_name, [])
    if not checks_for_scenario:
        print(f"\nSkipping {scenario_name}: no check mapping found in SCENARIO_CHECK_MAP")
        continue

    print(f"\n{'='*60}")
    print(f"Running: {scenario_name}")
    print(f"Check: {', '.join(checks_for_scenario)}")
    print(f"{'='*60}")

    try:
        test_run = okareo.run_simulation(
            target=target,
            driver=single_turn_driver,
            name=f"LLM04 Eval \u2014 {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="driver",
            scenario=scenario,
            max_turns=1,
            checks=checks_for_scenario,
        )
        all_results[scenario_name] = test_run
        print(f"  \u2713 Test run complete: {test_run.id}")
        if hasattr(test_run, "app_link") and test_run.app_link:
            print(f"  View: {test_run.app_link}")
    except Exception as e:
        print(f"  \u2717 Error: {e}")
        all_results[scenario_name] = None

### Results Summary

In [ ]:
print("\n" + "=" * 60)
print("LLM04 DATA AND MODEL POISONING \u2014 EVALUATION RESULTS")
print("OWASP Category: LLM04 | Risk Severity: High")
print("=" * 60)

print(f"\n{'Scenario':<42} {'Status':<10} {'Link / Run ID'}")
print("-" * 100)
for name, result in all_results.items():
    if result is None:
        print(f"{name:<42} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<42} {'COMPLETE':<10} {link}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal evaluated: {len(all_results)} | Errors: {errors}")
print(f"Check architecture: one model-based check per scenario")
if not errors:
    print("\u2713 All scenarios completed. See Okareo dashboard for full results.")

### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.

In [ ]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)